Analyse du lien entre statut et gravité, sur la durée des séjours

In [1]:
import pandas as pd

#importation des données sous forme de dataframes
MCO2022 = pd.read_csv("SAE/2022/MCO_2022r.csv", sep=";", encoding="latin-1")
Urg2022 = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
SSR2022 = pd.read_csv("SAE/2022/SSR_2022r.csv", sep=";", encoding="latin-1")

UrgP2022 = pd.read_csv("SAE/2022/URGENCES_P_2022a.csv", sep=";",encoding="latin-1")

FINESS = pd.read_excel("finess.xlsx")
FINESS = FINESS.drop(FINESS.columns[[1,2,4]],axis=1)
FINESS = FINESS.rename(columns={"FINESS":"FI","Statut Juridique":"Statut"})

#ajout d'une colonne aux données indiuant le statut de chaque établissement
MCO2022 = MCO2022.merge(FINESS,on='FI',how='left')
Urg2022 = Urg2022.merge(FINESS, on='FI', how='left')
SSR2022 = SSR2022.merge(FINESS, on='FI', how='left')

In [2]:
MCO2022.head(3)

,BOR,AN,FI,RS,FI_EJ,LIT_MED,JLI_MED,SEJHC_MED,SEJ0_MED,JOU_MED,...,DNEU,DPED,DOPH,ACTCLI_PM,ACTCLI_SAG,ACTTEC_PM,ACTTEC_DEN,ACTTEC_SAG,ACTTEC_PNM,Statut
0,MCO,2022,010000024,CH DE FLEYRIAT,010780054,270.0,92419.0,14972.0,4042.0,82796.0,...,105.0,105.0,NaN,92931.0,2523.0,63629.0,1204.0,5516.0,12582.0,Public
1,MCO,2022,010000032,CH BUGEY SUD,010780062,66.0,24090.0,4020.0,771.0,21185.0,...,90.0,24.0,NaN,17899.0,1822.0,21383.0,0.0,1676.0,7021.0,Public
2,MCO,2022,010000065,CH DE TREVOUX - MONTPENSIER,010780096,59.0,21535.0,1669.0,9.0,18131.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Public


In [3]:
hospidiag = pd.read_csv("Hospidiag/hd2022.csv", sep=";", encoding='latin-1')
hospidiag.head(3)

,finess,rs,champ_pmsi,taa,cat,taille_MCO,taille_M,taille_C,taille_O,A7,...,RH1,RH2,RH3,RH4,RH5,RH6,RH7,RH8,RH9,RH10
0,010007300,CLINIQUE AMBULATOIRE CENDANEG,OQN,TAA,CLI,T1,M1,C1,NaN,"15,2",...,NaN,"31004,3",53812,NaN,"0,3",NaN,NaN,.z,.z,.z
1,010007987,CH HAUTEVILLE,DGF,TAA,CH,T1,M1,C0,NaN,"27,7",...,NaN,NaN,NaN,"15,1",NaN,NaN,NaN,"7,5","42,8",NaN
2,010008407,CH DU HAUT BUGEY,DGF,TAA,CH,T2,M2,C2,O2,"7,2",...,29.0,"14727,9","54779,5","39,8","1,4","5,5",NaN,NaN,NaN,NaN


In [4]:
# Normalisation du finess 
MCO2022['FI'] = MCO2022['FI'].astype(str).str.zfill(9)
MCO2022['FI_EJ'] = MCO2022['FI_EJ'].astype(str).str.zfill(9)
hospidiag['finess'] = hospidiag['finess'].astype(str).str.zfill(9)

In [8]:
import numpy as np

MCO2022['Statut'] = MCO2022['Statut'].astype(str).str.strip()
condition_public = MCO2022['Statut'] == 'Public'

MCO2022['FI_Hospidiag'] = np.where(
    condition_public, 
    MCO2022['FI_EJ'],  # Valeur si Vrai (Public)
    MCO2022['FI']      # Valeur si Faux (Privé)
)

# On s'assure que cette nouvelle colonne est bien une chaîne de 9 caractères
MCO2022['FI_Hospidiag'] = MCO2022['FI_Hospidiag'].astype(str).str.split('.').str[0].str.zfill(9)

In [10]:
df_hosp_clean = hospidiag[['finess', 'A9']]

df_final = pd.merge(
    MCO2022, 
    df_hosp_clean, 
    left_on='FI_Hospidiag', 
    right_on='finess', 
    how='left'
)

df_final = df_final.drop(columns=['finess'])

In [30]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Copie et nettoyage initial
df_clean = df_final[df_final['SEJHC_MCO'] > 0].copy()

# 2. Nettoyage CRITIQUE du Statut AVANT la conversion en catégorie
# On convertit en string, on nettoie les espaces, et on remplace les chaines "nan" par de vrais NaN
df_clean['Statut'] = df_clean['Statut'].astype(str).str.strip()
df_clean['Statut'] = df_clean['Statut'].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})

# 3. Suppression des lignes vides (Statut OU A9)
df_clean = df_clean.dropna(subset=['Statut', 'A9'])

# 4. Conversion A9 et Calcul DMS
df_clean['A9'] = pd.to_numeric(df_clean['A9'].astype(str).str.replace(',', '.'))
df_clean['DMS'] = df_clean['JOU_MCO'] / df_clean['SEJHC_MCO']
df_clean['log_JOU_MCO'] = np.log1p(df_clean['JOU_MCO'])
df_clean['log_DMS'] = np.log1p(df_clean['DMS'])
df_clean['log_SEJHC_MCO'] = np.log1p(df_clean['SEJHC_MCO'])

# 5. Définir le Statut avec une référence explicite (ex: "Public")
# Cela permet de comparer tout le monde par rapport au Public
# "C(Statut, Treatment(reference='Public'))" dit à Python : "Public est mon zéro".
model = smf.ols("log_JOU_MCO ~ C(Statut, Treatment(reference='Public')) * A9 + log_SEJHC_MCO", data=df_clean).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            log_JOU_MCO   R-squared:                       0.913
Model:                            OLS   Adj. R-squared:                  0.912
Method:                 Least Squares   F-statistic:                     2416.
Date:                Tue, 24 Feb 2026   Prob (F-statistic):               0.00
Time:                        08:28:19   Log-Likelihood:                -804.85
No. Observations:                1393   AIC:                             1624.
Df Residuals:                    1386   BIC:                             1660.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                                                        coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

In [25]:
df_clean['A9'].describe()

count    1393.000000
mean       15.506095
std        16.495951
min         0.000000
25%         3.690000
50%        11.170000
75%        16.750000
max        94.740000
Name: A9, dtype: float64